# Session 10 — The model landscape: choosing a model
**Catalog · cost · speed · quality · the selection matrix**

The loan assistant does three different jobs. Each needs a different model. Your task: fill in the evidence, then choose one model per job and defend it.

Run top to bottom (Kernel → Restart & Run All). Before starting: `git pull`, `uv sync`, then `uv run jupyter lab module01/s10_model_matrix.ipynb`.

**How the decision is made (memorise this order):**
1) **hard constraints** eliminate models (data must stay in-house, latency limit, context too small); 
2) among survivors, a **weighted score** of cost, speed and quality ranks them; 
3) you **verify** the winner on your own data (S17 gives you the tool).

## 1. Setup
`landscape_utils.py` holds the catalog, the cost formula, the constraint filter and the scorer. `tokens_utils.py` (from S9b) gives us the two clients: Groq from `.env`, and Ollama on your laptop.

In [1]:
import sys; sys.path[:0] = [".", "module01", "../module01"]
import os
from landscape_utils import *
from tokens_utils import chat_client, ollama_client
ollama = ollama_client()
try:
    groq = chat_client()                      # Groq, from .env
except Exception as e:                        # no key yet: the notebook still runs; the hosted rows are just skipped
    groq = None
    print("no hosted client:", str(e)[:80])
GROQ_MODEL = os.getenv("MODEL")
print("hosted model:", GROQ_MODEL, "| expected EMI:", EXPECTED_EMI, "| expected ratio:", EXPECTED_RATIO, "%")

hosted model: openai/gpt-oss-20b | expected EMI: 17594.09 | expected ratio: 42.7 %


## 2. The catalog — ten rows, one table
Read `models_catalog.json` as a table. **Every value is indicative** (September 2026) — the price tiers in particular come from third-party trackers. `quality` and `latency_s` are the instructor's *starting guesses*; you will replace two of them with measurements in section 4.

Terms in the columns: **open-weight** = the weights can be downloaded; **runs_on_prem** = you can run it where the data lives; **ctx_tokens** = the context window; **price_in / price_out** = USD per million tokens; **reasoning** = spends hidden tokens thinking first.

In [2]:
cat = load_catalog()
print(f"{'model':46} {'access':11} {'open':5} {'ctx':>8} {'in$':>6} {'out$':>6} {'Q':>2} {'s':>4}")
for m in cat:
    pin = "?" if m["price_in"] is None else f"{m['price_in']:.3f}"
    pout = "?" if m["price_out"] is None else f"{m['price_out']:.2f}"
    print(f"{m['name'][:46]:46} {m['access']:11} {str(m['open_weight'])[:1]:5} {m['ctx_tokens']:8d} {pin:>6} {pout:>6} {m['quality']:2d} {m['latency_s']:4.1f}")

model                                          access      open       ctx    in$   out$  Q    s
gpt-oss-20b on Groq                            hosted-api  T       131072  0.075   0.30  3  1.5
gpt-oss-120b on Groq                           hosted-api  T       131072  0.150   0.60  4  2.0
Qwen3 32B on Groq                              hosted-api  T       131072  0.290   0.59  3  2.0
Llama 4 Scout 17B-16E on Groq                  hosted-api  T       131072  0.110   0.34  3  1.5
Llama 3.1 8B Instant on Groq                   hosted-api  T       131072  0.050      ?  2  1.0
llama3.2:3b on your laptop (Ollama)            local       T       131072  0.000   0.00  2  8.0
gpt-oss-20b self-hosted on a smaller GPU serve self-hosted T       131072  0.000   0.00  3  2.5
gpt-oss-120b self-hosted on your own GPU serve self-hosted T       131072  0.000   0.00  4  3.0
Frontier flagship (closed, vendor API)         vendor-api  F       200000  5.000  25.00  5  6.0
Frontier mid-tier (closed, vendor API)  

**Exercise 1 — fill the blank.** One row has `?` for output price. Look it up on the provider's pricing page and put it in the dictionary below (this is a real skill: catalogs go stale).

**Exercise 2 — verify two rows.** Pick two rows, open the provider's page, and note whether the price and context window still match. Write what you found in section 6.

*Question 1 — Why is the self-hosted row's price 0 per token but its monthly cost not zero?*

In [ ]:
OVERRIDES = {
    "groq-llama31-8b": {"price_out": '?'},      # <- exercise 1: replace ??? with the number you found
}

## 3. Three jobs, three bills
One product, three scenarios with different volumes and different rules. `SCENARIOS` holds them; edit any number to test "what if".

- **faq** — public product questions. Cheap and high-volume; no customer data.
- **documents** — reads a customer's salary slips and statements. **Customer PII: data must stay in-house.**
- **agent_assist** — live suggestions to a call-centre agent. **Must answer within about 3 seconds.**

In [8]:
for key, sc in SCENARIOS.items():
    print(f"\n{key}: {sc['title']}  ({sc['calls_per_day']:,} calls/day, {sc['in_tokens']:,} in / {sc['out_tokens']} out per call)")
    for m in cat:
        m = {**m, **OVERRIDES.get(m["id"], {})}
        inr = monthly_cost_inr(m, sc)
        print(f"   {m['name'][:46]:46} " + ("price unknown" if inr is None else f"Rs {inr:>12,}/month"))


faq: Public product FAQ chatbot  (20,000 calls/day, 1,200 in / 200 out per call)
   gpt-oss-20b on Groq                            Rs        7,560/month
   gpt-oss-120b on Groq                           Rs       15,120/month
   Qwen3 32B on Groq                              Rs       23,486/month
   Llama 4 Scout 17B-16E on Groq                  Rs       10,080/month


TypeError: unsupported operand type(s) for +: 'float' and 'str'

*Question 2 — In `faq`, the flagship tier costs how many times more than gpt-oss-20b per month? In which scenario does that gap matter least, and why?*

## 4. Measure — the same prompt on a hosted model and on your laptop
Catalog numbers are a prior; **your measurements are evidence.** We send one prompt to Groq and to Ollama, time it, and check the arithmetic.

- **TTFT** — time to first token you can see. For a reasoning model this includes the hidden thinking.
- **tokens/s** — output speed.
- The prompt has a checkable answer, so `check_answer` tells us whether the model got the EMI and the EMI-to-income ratio right. That is a tiny **evaluation**; S17 builds the full version.

In [ ]:
print(PROMPT, "\n")
RESULTS = {}
for label, client, model in [("groq", groq, GROQ_MODEL), ("ollama", ollama, "llama3.2:3b")]:
    if client is None:
        print(f"--- {label}: skipped (no client - add your key to .env)\n"); continue
    try:
        r = time_stream(client, model)
        r["check"] = check_answer(r["text"])
        RESULTS[label] = r
        print(f"--- {label}: {model}")
        print(f"first token {r['ttft_s']} s | total {r['total_s']} s | {r['completion_tokens']} tokens | {r['tokens_per_s']} tokens/s")
        print("arithmetic check:", r["check"])
        print(r["text"].strip()[:500], "\n")
    except Exception as e:
        print(f"--- {label}: skipped ({type(e).__name__}: {str(e)[:120]})\n")

**Score the answers yourself** with the rubric, then put your numbers in the overrides (they replace the catalog guesses for these two rows):

In [ ]:
print(RUBRIC)

In [ ]:
# Fill in from YOUR run. total_s is a good stand-in for latency_s. quality: 1-5 from the rubric.
MEASURED = {
    # "groq-gpt-oss-20b":   {"latency_s": 1.4, "quality": 4},
    # "ollama-llama32-3b":  {"latency_s": 9.0, "quality": 2},
}
OVERRIDES.update(MEASURED)
OVERRIDES

*Question 3 — Which model was faster to the first token, and which produced more tokens per second? Were they the same one? What does that tell you about "fast"?*

## 5. The selection matrix
For each scenario: **hard constraints first** (a model that fails one is out, however cheap), then a weighted score of cost, speed and quality. Weights are in `SCENARIOS`; change them and watch the winner move.

In [ ]:
for key, sc in SCENARIOS.items():
    ranked, excluded = rank(sc, overrides=OVERRIDES)
    print(f"\n=== {key}: {sc['title']}   weights {sc['weights']}")
    if not ranked:
        print("   nobody survives the hard constraints - relax one or add a model")
    for i, m in enumerate(ranked, 1):
        print(f"  {i}. {m['name'][:46]:46} score {m['score']:5.1f}   Rs {m['cost_inr']:>10,}/month   {m['latency_s']} s   Q{m['quality']}")
    for name, why in excluded:
        print(f"  x  {name[:46]:46} {'; '.join(why)}")

*Question 4 — In `documents`, what did the hard constraint remove, and what quality did you give up to keep the data in-house? What would you do about it?*

*Question 5 — Change the `faq` weights to quality 0.7, cost 0.1. Does the winner change? What does that tell you about who should choose the weights?*

## 6. Your decision — this is what you submit
For each scenario write: the model you would choose, the runner-up, and **what evidence would change your mind**. Also write what you found when you verified two catalog rows (exercise 2), and answers to questions 1–5.

**faq:** 

**documents:** 

**agent_assist:** 

**Verified rows (exercise 2):** 

**Q1–Q5:** 

Then save and submit: run the cell below, and upload `module01/s10_matrix_output.md` through the submission form (personal repositories start at Session 14).

In [ ]:
from pathlib import Path
import landscape_utils
OUT = Path(landscape_utils.__file__).with_name("s10_matrix_output.md")   # next to the helper, whichever folder Jupyter runs in
lines = ["# Session 10 - selection matrix output\n", f"expected EMI {EXPECTED_EMI}, ratio {EXPECTED_RATIO}%\n"]
for key, sc in SCENARIOS.items():
    ranked, excluded = rank(sc, overrides=OVERRIDES)
    lines.append(f"\n## {key}: {sc['title']}")
    for i, m in enumerate(ranked, 1):
        lines.append(f"{i}. {m['name']} - score {m['score']} - Rs {m['cost_inr']:,}/month - {m['latency_s']} s - Q{m['quality']}")
    for name, why in excluded:
        lines.append(f"- excluded: {name} ({'; '.join(why)})")
lines.append("\n## Measured\n" + str({k: {kk: v[kk] for kk in ('ttft_s','total_s','tokens_per_s','check')} for k, v in RESULTS.items()}))
OUT.write_text("\n".join(lines), encoding="utf-8")
print("saved", OUT)